# Different Options for Querying Files in Databricks

Databricks provides multiple ways to query data files without creating persistent tables. Here are the main approaches:

---

## 1. **`read_files()` Function** (Recommended for Ad-hoc Queries)

**Direct querying** without creating any objects:

```sql
SELECT * FROM read_files(
  'path/to/files',
  format => 'csv',
  header => true
) LIMIT 10;
```

**Benefits:**
* No table creation needed
* Works with wildcards (`*.json`, `*.parquet`)
* Supports all formats: CSV, JSON, Parquet, Delta, Avro, ORC
* Auto-detects compression (`.gz`, `.bz2`)
* Can specify options inline

---

## 2. **CTAS (Create Table As Select)** (Recommended for Production)

**Create persistent Delta tables** from files:

```sql
CREATE OR REPLACE TABLE catalog.schema.table_name AS
SELECT * FROM read_files('path/to/files', format => 'csv');
```

**Benefits:**
* Creates optimized Delta Lake tables
* Better performance for repeated queries
* Supports Unity Catalog governance
* Data is copied into managed storage

---

## 3. **Temporary Views with `USING` Format**

**Create temporary views** that reference external files:

```sql
CREATE OR REPLACE TEMP VIEW view_name
USING JSON
OPTIONS (path "path/to/files/*.json");

SELECT * FROM view_name;
```

**Benefits:**
* Session-scoped (disappears after session ends)
* No data copying
* Good for exploratory work
* Supports wildcards

---

## 4. **Direct Path Syntax** (Legacy)

**Query files using backtick notation:**

```sql
SELECT * FROM json.`path/to/file.json`;
SELECT * FROM parquet.`path/to/*.parquet`;
SELECT * FROM delta.`path/to/delta_table/`;
```

**Benefits:**
* Shortest syntax
* Auto-detects format from extension
* No options supported (limited flexibility)

---

## Comparison Table

| Method | Persistent | UC Governance | Performance | Options Support | Best For |
|--------|-----------|---------------|-------------|-----------------|----------|
| **read_files()** | No | No | Medium | ✅ Full | Ad-hoc queries, exploration |
| **CTAS** | Yes (Delta) | ✅ Yes | ✅ High | ✅ Full | Production tables |
| **TEMP VIEW** | No (session) | No | Medium | ✅ Full | Temporary work |
| **Direct Path** | No | No | Medium | ❌ Limited | Quick checks |

---

## When to Use Each

* **Exploring data**: Use `read_files()` or direct path syntax
* **Building production pipelines**: Use CTAS to create Delta tables
* **Temporary transformations**: Use TEMP VIEWs
* **Quick inspection**: Use direct path syntax (e.g., `json.\`path\``)

---

## Supported File Formats

Databricks supports a wide range of file formats for reading and querying:

### Structured Formats
* **CSV** — Comma-separated values with header and delimiter options
* **JSON** — JavaScript Object Notation, supports nested structures
* **Parquet** — Columnar format, optimized for analytics (most efficient)
* **ORC** — Optimized Row Columnar format
* **Avro** — Row-based format with schema evolution support
* **Delta** — Open-source lakehouse format (best for production)

### Text Formats
* **TEXT** — Plain text files (one line per record)
* **BINARYFILE** — Read entire files as binary content

### Specialized Formats
* **XML** — Extensible Markup Language (requires spark-xml library)

### Compression Support

All formats support automatic compression detection:
* **GZIP** (`.gz`) — Good compression, slower
* **BZIP2** (`.bz2`) — Better compression, slowest
* **Snappy** (`.snappy`) — Fast, moderate compression
* **LZ4** (`.lz4`) — Very fast, moderate compression
* **ZSTD** (`.zst`) — Balanced compression and speed

### Format-Specific Options

**CSV Options:**
```sql
format => 'csv',
header => true,
delimiter => ',',
inferSchema => true
```

**JSON Options:**
```sql
format => 'json',
multiLine => true  -- For multi-line JSON objects
```

**Parquet/Delta:**
```sql
format => 'parquet'  -- No additional options needed, auto-optimized
```

In [0]:
%python
display(dbutils.fs.ls("/databricks-datasets/"))

In [0]:
-- Preview the online retail CSV file directly without creating a table
SELECT * 
FROM read_files(
  'dbfs:/databricks-datasets/online_retail/data-001/data.csv',
  format => 'csv',
  header => true
)
LIMIT 10;

-- Tbale can be created using CTAS for external data ingestion from a well defined schema sources such as parquet

In [0]:
-- Create Delta table from CSV using CTAS (Create Table As Select)
CREATE OR REPLACE TABLE workspace.default.online_retail AS
SELECT * FROM read_files(
  'dbfs:/databricks-datasets/online_retail/data-001/data.csv',
  format => 'csv',
  header => true
);

-- Verify the table was created and show row count
SELECT COUNT(*) as total_rows FROM workspace.default.online_retail;

In [0]:
-- Sample queries on the online retail table

-- Show sample records
SELECT * FROM workspace.default.online_retail LIMIT 5;

-- Get top 10 countries by number of orders
SELECT Country, COUNT(*) as order_count
FROM workspace.default.online_retail
GROUP BY Country
ORDER BY order_count DESC
LIMIT 10;

In [0]:
%python
# Explore IOT stream dataset for JSON files
iot_files = dbutils.fs.ls("/databricks-datasets/iot-stream/data-device/")
display(iot_files[:5])

# Check samples directory
samples_files = dbutils.fs.ls("/databricks-datasets/samples/")
display(samples_files)

In [0]:
-- Preview IOT device JSON data (gzipped JSON files)
SELECT * 
FROM read_files(
  'dbfs:/databricks-datasets/iot-stream/data-device/*.json.gz',
  format => 'json'
)
LIMIT 10;

In [0]:
-- Create a managed Delta table from IOT device JSON
CREATE OR REPLACE TEMP VIEW iot_devices1
USING JSON
OPTIONS (
  path "dbfs:/databricks-datasets/iot-stream/data-device/*.json.gz"
);


SELECT * FROM iot_devices1 LIMIT 5;

In [0]:
-- Another example: Preview people.json (simpler structure)
SELECT * 
FROM JSON.`dbfs:/databricks-datasets/samples/people/people.json`; -- note the back ticks not quotes

In [0]:
CREATE OR REPLACE TEMP VIEW iot_devices2
USING JSON
OPTIONS (
  path "dbfs:/databricks-datasets/iot-stream/data-device/*.json.gz")

In [0]:
--The read_files function automatically tries to infer a unified schema from all the source files. If any value doesn’t match the expected schema, it's stored in an extra column called _rescued_data as a JSON string.
select *
FROM workspace.default.online_retail
WHERE _rescued_data IS NOT NULL;

In [0]:
SELECT *, _metadata.file_path, _metadata.file_modification_time, _metadata.file_size, _metadata.file_block_start, _metadata.file_block_length
FROM JSON.`dbfs:/databricks-datasets/samples/people/people.json`;

In [0]:
SELECT *, _metadata
FROM JSON.`dbfs:/databricks-datasets/samples/people/people.json`;

## `_metadata` Column - Available Fields

When reading files in Databricks, the `_metadata` pseudo-column provides file-level information for tracking data lineage, debugging, and incremental processing.

---

### Available Fields

| Field | Type | Description |
|-------|------|-------------|
| **`_metadata.file_path`** | STRING | Full path to the source file |
| **`_metadata.file_name`** | STRING | Name of the file (without directory path) |
| **`_metadata.file_size`** | LONG | Size of the file in bytes |
| **`_metadata.file_modification_time`** | TIMESTAMP | Last modification timestamp of the file |
| **`_metadata.file_block_start`** | LONG | Starting byte position of the block being read |
| **`_metadata.file_block_length`** | LONG | Length of the block in bytes |

---

### Basic Usage

```sql
SELECT 
  *,
  _metadata.file_path,
  _metadata.file_name,
  _metadata.file_size,
  _metadata.file_modification_time
FROM read_files('path/*.json', format => 'json');
```

---

### When is `_metadata` Available?

✅ **Available when:**
* Using `read_files()` function
* Using direct path syntax (e.g., `json.\`path\``)
* Using TEMP VIEWs created with `USING format OPTIONS (path ...)`

❌ **NOT available when:**
* Querying Delta tables created via CTAS
* Querying managed Unity Catalog tables (unless external tables pointing to files)

---

### Common Use Cases

* **Data Lineage** — Track which file each record came from
* **Incremental Processing** — Filter records by file modification time
* **File Size Analysis** — Identify large files causing performance issues
* **Debugging** — Understand block-level distribution for large files

In [0]:
-- Common _metadata Use Cases

-- 1. Track Data Lineage - See which file each record came from
SELECT 
  Country,
  COUNT(*) as record_count,
  _metadata.file_path as source_file
FROM read_files(
  'dbfs:/databricks-datasets/online_retail/data-001/*.csv',
  format => 'csv',
  header => true
)
GROUP BY Country, _metadata.file_path
ORDER BY record_count DESC
LIMIT 10;

-- 2. File Size Analysis - Identify large files
SELECT 
  _metadata.file_path,
  _metadata.file_size / 1024 / 1024 as size_mb,
  COUNT(*) as record_count
FROM read_files(
  'dbfs:/databricks-datasets/iot-stream/data-device/*.json.gz',
  format => 'json'
)
GROUP BY _metadata.file_path, _metadata.file_size
ORDER BY size_mb DESC;

-- 3. Incremental Processing - Filter by modification time
SELECT 
  *,
  _metadata.file_modification_time
FROM read_files(
  'dbfs:/databricks-datasets/online_retail/data-001/*.csv',
  format => 'csv',
  header => true
)
WHERE _metadata.file_modification_time > '2020-01-01'
LIMIT 5;